In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2
        ),
        "GiB"
    )


PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GiB


In [ ]:
%pip install -q -U transformers peft trl datasets accelerate scikit-learn pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.


In [ ]:
%pip install -q "datasets==4.8.0" "pyarrow==23.0.1" "pandas==2.2.3"

In [ ]:
import torch
import pandas
import pyarrow
import datasets
import transformers
import peft
import trl
import accelerate

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())
print("Datasets:", datasets.__version__)
print("PyArrow:", pyarrow.__version__)
print("Pandas:", pandas.__version__)
print("TRL:", trl.__version__)

# Functional dataset test
from datasets import Dataset

sample = Dataset.from_dict({
    "text": ["AI model released", "Football championship"],
    "label": [3, 1]
})

print("\nDataset test:", sample[0])


GPU: Tesla T4
CUDA: True
Datasets: 4.8.0
PyArrow: 23.0.1
Pandas: 2.2.3
TRL: 1.18.1

Dataset test: {'text': 'AI model released', 'label': 3}


In [ ]:
import torch
import transformers
import peft
import trl
import datasets
import accelerate
import sklearn
import pandas as pd

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)

print("\nGPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.14.1+cu130
Transformers: 5.23.0
PEFT: 0.21.3
TRL: 1.18.1
Datasets: 4.8.0
Accelerate: 1.16.0

GPU: Tesla T4
CUDA available: True


##2A — Load the AG News dataset

In [ ]:
from datasets import load_dataset

DATASET_NAME = "fancyzhx/ag_news"

train_data = load_dataset(DATASET_NAME, split="train")
test_data = load_dataset(DATASET_NAME, split="test")

print("Training examples:", len(train_data))
print("Test examples:", len(test_data))
print("Features:", train_data.features)

print("\nSample example:")
print(train_data[0])

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Training examples: 120000
Test examples: 7600
Features: {'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}

Sample example:
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


#2B — Create our balanced experimental dataset

In [ ]:
import random
from collections import Counter, defaultdict

SEED = 42

def group_indices_by_label(dataset):
    grouped = defaultdict(list)
    for index, label in enumerate(dataset["label"]):
        grouped[int(label)].append(index)
    return grouped

train_groups = group_indices_by_label(train_data)
test_groups = group_indices_by_label(test_data)

train_indices = []
val_indices = []
test_indices = []

for label in range(4):
    # Shuffle training candidates deterministically
    candidates = train_groups[label].copy()
    random.Random(SEED + label).shuffle(candidates)

    train_indices.extend(candidates[:200])
    val_indices.extend(candidates[200:225])

    # Use a separate official test dataset
    test_candidates = test_groups[label].copy()
    random.Random(SEED + 100 + label).shuffle(test_candidates)

    test_indices.extend(test_candidates[:25])

train_split = train_data.select(train_indices).shuffle(seed=SEED)
val_split = train_data.select(val_indices).shuffle(seed=SEED)
test_split = test_data.select(test_indices).shuffle(seed=SEED)

assert set(train_indices).isdisjoint(set(val_indices))

print("Train:", len(train_split))
print("Validation:", len(val_split))
print("Test:", len(test_split))

print("\nTraining class distribution:")
print(Counter(train_split["label"]))

print("\nValidation class distribution:")
print(Counter(val_split["label"]))

print("\nTest class distribution:")
print(Counter(test_split["label"]))

Train: 800
Validation: 100
Test: 100

Training class distribution:
Counter({0: 200, 3: 200, 2: 200, 1: 200})

Validation class distribution:
Counter({2: 25, 0: 25, 1: 25, 3: 25})

Test class distribution:
Counter({2: 25, 0: 25, 1: 25, 3: 25})


#2C — Define our classification instruction

In [ ]:
LABEL_MAP = {
    0: "WORLD",
    1: "SPORTS",
    2: "BUSINESS",
    3: "SCI_TECH",
}

SYSTEM_INSTRUCTION = (
    "You are a news classification assistant. "
    "Classify each news article into exactly one category: "
    "WORLD, SPORTS, BUSINESS, or SCI_TECH. "
    "Return only the category label, without explanation."
)

def make_messages(article_text):
    return [
        {
            "role": "system",
            "content": SYSTEM_INSTRUCTION
        },
        {
            "role": "user",
            "content": f"News article:\n{article_text}"
        }
    ]

sample = train_split[0]

print("Input messages:")
print(make_messages(sample["text"]))

print("\nExpected classification:")
print(LABEL_MAP[sample["label"]])

Input messages:
[{'role': 'system', 'content': 'You are a news classification assistant. Classify each news article into exactly one category: WORLD, SPORTS, BUSINESS, or SCI_TECH. Return only the category label, without explanation.'}, {'role': 'user', 'content': "News article:\nSponsorship not mentioned for years in department reviews: auditor general (Canadian Press) Canadian Press - OTTAWA (CP) - The federal sponsorship program operated for years without ever being mentioned in the Public Works Department's annual performance reviews, Auditor General Sheila Fraser said Tuesday."}]

Expected classification:
WORLD


#3A-Load the original Qwen model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

base_model.eval()

total_params = sum(p.numel() for p in base_model.parameters())

print("Model:", MODEL_NAME)
print("Parameters:", f"{total_params:,}")
print("Model device:", base_model.device)
print("Model dtype:", next(base_model.parameters()).dtype)
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GiB"
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model: Qwen/Qwen2.5-0.5B-Instruct
Parameters: 494,032,768
Model device: cuda:0
Model dtype: torch.float16
GPU memory allocated: 0.92 GiB


#3B — Run our first classification prediction

In [ ]:
ARTICLE_TOKEN_LIMIT = 160

def prepare_model_input(example):
    # Limit article length without removing the chat template.
    article_tokens = tokenizer(
        example["text"],
        add_special_tokens=False
    )["input_ids"][:ARTICLE_TOKEN_LIMIT]

    short_article = tokenizer.decode(
        article_tokens,
        skip_special_tokens=True
    )

    messages = make_messages(short_article)

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    return inputs.to(base_model.device)


def predict_label(model, example):
    inputs = prepare_model_input(example)
    prompt_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0, prompt_length:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()


sample = test_split[0]

prediction = predict_label(base_model, sample)

print("Article:", sample["text"][:250])
print("\nExpected:", LABEL_MAP[sample["label"]])
print("Predicted:", prediction)
print("Correct:", prediction == LABEL_MAP[sample["label"]])

Article: National Business Briefs ALEXANDRIA, Va. - Google Inc. won a major legal victory yesterday when a federal judge ruled that its advertising policy does not violate federal trademark laws.

Expected: BUSINESS
Predicted: WORLD
Correct: False


#3C — Evaluate the base model on 100 examples

In [ ]:
import os
import json
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

RESULTS_DIR = "task1_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

VALID_LABELS = list(LABEL_MAP.values())

base_model.eval()

records = []

for example in tqdm(test_split, desc="Base model evaluation"):

    expected = LABEL_MAP[int(example["label"])]

    raw_prediction = predict_label(
        base_model,
        example
    )

    # Strict instruction-following evaluation:
    # Only the four exact labels are valid.
    predicted = (
        raw_prediction
        if raw_prediction in VALID_LABELS
        else "INVALID"
    )

    records.append({
        "article": example["text"],
        "expected_label": expected,
        "raw_prediction": raw_prediction,
        "predicted_label": predicted,
        "correct": predicted == expected
    })

base_df = pd.DataFrame(records)

y_true = base_df["expected_label"]
y_pred = base_df["predicted_label"]

metrics = {
    "model": MODEL_NAME,
    "evaluation_examples": len(base_df),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_f1": float(
        f1_score(
            y_true,
            y_pred,
            labels=VALID_LABELS,
            average="macro",
            zero_division=0
        )
    ),
    "correct_predictions": int(base_df["correct"].sum()),
    "invalid_outputs": int(
        (base_df["predicted_label"] == "INVALID").sum()
    )
}

# Save predictions
base_df.to_csv(
    f"{RESULTS_DIR}/base_predictions.csv",
    index=False
)

# Save evaluation metrics
with open(
    f"{RESULTS_DIR}/baseline_metrics.json",
    "w"
) as f:
    json.dump(metrics, f, indent=2)

print("\nBASE MODEL RESULTS")
print(json.dumps(metrics, indent=2))

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        zero_division=0
    )
)

print("\nSaved:")
print(f"{RESULTS_DIR}/base_predictions.csv")
print(f"{RESULTS_DIR}/baseline_metrics.json")

Base model evaluation:   0%|          | 0/100 [00:00<?, ?it/s]


BASE MODEL RESULTS
{
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "evaluation_examples": 100,
  "accuracy": 0.28,
  "macro_f1": 0.20582357978875251,
  "correct_predictions": 28,
  "invalid_outputs": 0
}

CLASSIFICATION REPORT
              precision    recall  f1-score   support

       WORLD       0.24      0.84      0.37        25
      SPORTS       0.83      0.20      0.32        25
    BUSINESS       0.00      0.00      0.00        25
    SCI_TECH       0.33      0.08      0.13        25

    accuracy                           0.28       100
   macro avg       0.35      0.28      0.21       100
weighted avg       0.35      0.28      0.21       100


Saved:
task1_results/base_predictions.csv
task1_results/baseline_metrics.json


#4 — Train the LoRA adapter

#4A — Prepare training and validation datasets

In [ ]:
from datasets import Dataset

def build_training_example(example):
    # Identical article truncation to baseline inference
    article_ids = tokenizer(
        example["text"],
        add_special_tokens=False
    )["input_ids"][:ARTICLE_TOKEN_LIMIT]

    article = tokenizer.decode(
        article_ids,
        skip_special_tokens=True
    )

    messages = make_messages(article)

    # Full system + user prompt, ending at assistant start
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Correct answer, followed by Qwen's end-of-turn token
    completion = (
        LABEL_MAP[int(example["label"])]
        + tokenizer.eos_token
    )

    return {
        "prompt": prompt,
        "completion": completion
    }


train_sft = Dataset.from_list([
    build_training_example(x) for x in train_split
])

val_sft = Dataset.from_list([
    build_training_example(x) for x in val_split
])

print("Training examples:", len(train_sft))
print("Validation examples:", len(val_sft))

print("\nSample prompt:")
print(train_sft[0]["prompt"][-500:])

print("\nExpected completion:")
print(repr(train_sft[0]["completion"]))

# Check that the complete sequences fit our training limit
max_tokens = max(
    len(
        tokenizer(
            x["prompt"] + x["completion"],
            add_special_tokens=False
        )["input_ids"]
    )
    for x in train_sft
)

print("\nMaximum training sequence length:", max_tokens)

assert max_tokens <= 256, (
    "Some examples exceed 256 tokens. "
    "Increase the training max_length before continuing."
)

Training examples: 800
Validation examples: 100

Sample prompt:
 into exactly one category: WORLD, SPORTS, BUSINESS, or SCI_TECH. Return only the category label, without explanation.<|im_end|>
<|im_start|>user
News article:
Sponsorship not mentioned for years in department reviews: auditor general (Canadian Press) Canadian Press - OTTAWA (CP) - The federal sponsorship program operated for years without ever being mentioned in the Public Works Department's annual performance reviews, Auditor General Sheila Fraser said Tuesday.<|im_end|>
<|im_start|>assistant


Expected completion:
'WORLD<|im_end|>'

Maximum training sequence length: 218


#4B — Configure LoRA

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print(lora_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.3', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'v_proj', 'q_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)


#4C — Configure and run training

In [ ]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
from peft.import_utils import is_torchao_available

print("TorchAO available:", is_torchao_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Training examples:", len(train_sft))

TorchAO available: False
GPU: Tesla T4
Training examples: 800


In [ ]:
import os
import json
import torch
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = "task1_results"

training_args = SFTConfig(
    output_dir=f"{OUTPUT_DIR}/checkpoints",

    # Two short training epochs
    num_train_epochs=2,

    # Optimizer
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_steps=5,
    optim="adamw_torch",

    # Effective batch size = 4 × 4 = 16
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,

    # Precision and memory
    fp16=True,
    bf16=False,
    gradient_checkpointing=False,
    max_length=256,
    packing=False,

    # Train only on the correct completion
    completion_only_loss=True,

    # Evaluation and logging
    eval_strategy="epoch",
    logging_steps=10,
    save_strategy="no",
    report_to="none",

    seed=42
)

trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_sft,
    eval_dataset=val_sft,
    processing_class=tokenizer,
    peft_config=lora_config
)

print("Trainable parameter summary:")
trainer.model.print_trainable_parameters()

# Confirm the dataset contains supervised target tokens
sample_batch = trainer.data_collator([
    trainer.train_dataset[0]
])

assert (sample_batch["labels"] != -100).any(), (
    "No supervised completion tokens found."
)

print("\nCompletion-only supervision check passed.")

print("\nStarting LoRA training...")
torch.cuda.reset_peak_memory_stats()

train_result = trainer.train()

# Save the trained adapter, not the full base model
ADAPTER_DIR = f"{OUTPUT_DIR}/lora_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Preserve training and validation logs
with open(f"{OUTPUT_DIR}/training_logs.json", "w") as f:
    json.dump(
        trainer.state.log_history,
        f,
        indent=2,
        default=str
    )

print("\nTraining completed.")
print("Training loss:", train_result.training_loss)
print(
    "Peak PyTorch GPU memory:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GiB"
)
print("Adapter saved to:", ADAPTER_DIR)

/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

... (4 outputs left out)

Dropping fully masked examples from eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Trainable parameter summary:
trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093

Completion-only supervision check passed.

Starting LoRA training...


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.190678,0.201593,0.241204,0.928815,88065.000000
2,0.132154,0.172618,0.174768,0.943671,176130.000000



Training completed.
Training loss: 0.21561040997505188
Peak PyTorch GPU memory: 1.95 GiB
Adapter saved to: task1_results/lora_adapter


#5A — Evaluate the fine-tuned model

In [ ]:
import os
import json
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

RESULTS_DIR = "task1_results"

# 1. Load saved baseline evidence
baseline_path = f"{RESULTS_DIR}/base_predictions.csv"

base_df = pd.read_csv(baseline_path)

# Verify both evaluations use the identical test set.
assert len(base_df) == len(test_split) == 100

assert base_df["article"].tolist() == [
    example["text"] for example in test_split
], "Test data mismatch: stop and inspect the dataset!"

# 2. Use the trained LoRA model
lora_model = trainer.model
lora_model.eval()

VALID_LABELS = list(LABEL_MAP.values())

records = []

# 3. Run the held-out evaluation
for example in tqdm(
    test_split,
    desc="LoRA model evaluation"
):
    expected = LABEL_MAP[int(example["label"])]

    raw_prediction = predict_label(
        lora_model,
        example
    )

    predicted = (
        raw_prediction
        if raw_prediction in VALID_LABELS
        else "INVALID"
    )

    records.append({
        "article": example["text"],
        "expected_label": expected,
        "raw_prediction": raw_prediction,
        "predicted_label": predicted,
        "correct": predicted == expected
    })

lora_df = pd.DataFrame(records)

# 4. Calculate the same metrics as the baseline
lora_metrics = {
    "model": MODEL_NAME + " + LoRA",
    "evaluation_examples": len(lora_df),
    "accuracy": float(
        accuracy_score(
            lora_df["expected_label"],
            lora_df["predicted_label"]
        )
    ),
    "macro_f1": float(
        f1_score(
            lora_df["expected_label"],
            lora_df["predicted_label"],
            labels=VALID_LABELS,
            average="macro",
            zero_division=0
        )
    ),
    "correct_predictions": int(
        lora_df["correct"].sum()
    ),
    "invalid_outputs": int(
        (lora_df["predicted_label"] == "INVALID").sum()
    )
}

# 5. Save results
lora_df.to_csv(
    f"{RESULTS_DIR}/lora_predictions.csv",
    index=False
)

with open(
    f"{RESULTS_DIR}/finetuned_metrics.json",
    "w"
) as f:
    json.dump(lora_metrics, f, indent=2)

# 6. Display results
print("\nFINE-TUNED MODEL RESULTS")
print(json.dumps(lora_metrics, indent=2))

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        lora_df["expected_label"],
        lora_df["predicted_label"],
        labels=VALID_LABELS,
        zero_division=0
    )
)

# 7. Direct comparison
with open(
    f"{RESULTS_DIR}/baseline_metrics.json"
) as f:
    baseline_metrics = json.load(f)

print("\n========== BASE vs LoRA ==========")
print(
    f"Accuracy: "
    f"{baseline_metrics['accuracy']:.1%} -> "
    f"{lora_metrics['accuracy']:.1%}"
)
print(
    f"Macro-F1: "
    f"{baseline_metrics['macro_f1']:.4f} -> "
    f"{lora_metrics['macro_f1']:.4f}"
)
print(
    f"Accuracy difference: "
    f"{(lora_metrics['accuracy'] - baseline_metrics['accuracy']) * 100:+.1f} "
    "percentage points"
)

LoRA model evaluation:   0%|          | 0/100 [00:00<?, ?it/s]


FINE-TUNED MODEL RESULTS
{
  "model": "Qwen/Qwen2.5-0.5B-Instruct + LoRA",
  "evaluation_examples": 100,
  "accuracy": 0.84,
  "macro_f1": 0.8383480248783577,
  "correct_predictions": 84,
  "invalid_outputs": 0
}

CLASSIFICATION REPORT
              precision    recall  f1-score   support

       WORLD       0.77      0.68      0.72        25
      SPORTS       0.96      1.00      0.98        25
    BUSINESS       0.88      0.84      0.86        25
    SCI_TECH       0.75      0.84      0.79        25

    accuracy                           0.84       100
   macro avg       0.84      0.84      0.84       100
weighted avg       0.84      0.84      0.84       100


========== BASE vs LoRA ==========
Accuracy: 28.0% -> 84.0%
Macro-F1: 0.2058 -> 0.8383
Accuracy difference: +56.0 percentage points


#5B — Analyze three real fine-tuning failures

##Step 1 — Back up the experiment

In [ ]:
import os
import shutil
from google.colab import files

assert os.path.exists(
    "task1_results/lora_adapter/adapter_config.json"
), "LoRA adapter not found"

archive = shutil.make_archive(
    "/content/task1_backup",
    "zip",
    root_dir="task1_results"
)

print("Backup created:", archive)
files.download(archive)

Backup created: /content/task1_backup.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Step 2 — Extract the 16 incorrect predictions

In [ ]:
import pandas as pd

# Keep only the examples LoRA classified incorrectly.
errors = lora_df.loc[~lora_df["correct"]].copy()

# Preserve original test-set index for comparison.
errors.insert(0, "test_index", errors.index)

# Attach the original model's prediction.
errors["base_prediction"] = base_df.loc[
    errors.index, "predicted_label"
].values

# Save all errors.
errors.to_csv(
    "task1_results/all_failure_cases.csv",
    index=False
)

# Choose 3 examples, preferring different true categories.
selected = (
    errors.groupby("expected_label", sort=False)
    .head(1)
    .head(3)
    .copy()
)

# We'll write explanations after examining the articles.
selected["failure_analysis"] = ""

selected.to_csv(
    "task1_results/failure_cases.csv",
    index=False
)

print("Total LoRA failures:", len(errors))
print("\nFailures by actual category:")
print(errors["expected_label"].value_counts())

print("\n========== THREE FAILURE CASES ==========")

for _, row in selected.iterrows():
    print("\n" + "=" * 65)
    print("Test index:", row["test_index"])
    print("Article:", row["article"][:600])
    print("Correct label:", row["expected_label"])
    print("Base prediction:", row["base_prediction"])
    print("LoRA prediction:", row["predicted_label"])

Total LoRA failures: 16

Failures by actual category:
expected_label
WORLD       8
BUSINESS    4
SCI_TECH    4
Name: count, dtype: int64

========== THREE FAILURE CASES ==========

Test index: 12
Article: Malaysia discovers new bird flu case KUALA LUMPUR - Malaysia discovered new cases of bird flu on Sunday within a quarantine area in northern Malaysia where workers have struggled for a month to eradicate the virus.
Correct label: WORLD
Base prediction: WORLD
LoRA prediction: SCI_TECH

Test index: 25
Article: Retailers Looking to Move Plasma TV's (AP) AP - Hanging stockings by the chimney with care? Retailers hope that St. Nicholas soon will be there  #151; to hang a 42-inch plasma-screen TV.
Correct label: BUSINESS
Base prediction: WORLD
LoRA prediction: SCI_TECH

Test index: 29
Article: Southern Africa Faces Food, Water Crises - Study (Reuters) Reuters - Southern Africa faces major\challenges to feed its swelling populations and to keep its\wells from running dry, a study showed Wedn

##Saving the failure explanations to the experiment

In [ ]:
failure_notes = {
    12: (
        "The article concerns a bird flu outbreak in Malaysia. "
        "The model predicted SCI_TECH instead of WORLD, possibly "
        "emphasizing the medical subject over its international-news framing."
    ),
    25: (
        "The article describes retailers promoting plasma televisions. "
        "The model predicted SCI_TECH instead of BUSINESS, possibly "
        "emphasizing the technology product over the retail sales context."
    ),
    29: (
        "The article describes a study about food and water challenges "
        "in Southern Africa. The model predicted WORLD instead of SCI_TECH, "
        "possibly emphasizing regional affairs over the research framing."
    ),
}

selected["failure_analysis"] = (
    selected["test_index"].map(failure_notes)
)

assert selected["failure_analysis"].notna().all()

selected.to_csv(
    "task1_results/failure_cases.csv",
    index=False
)

print(
    selected[
        [
            "test_index",
            "expected_label",
            "predicted_label",
            "failure_analysis"
        ]
    ].to_string(index=False)
)

print("\nSaved three documented failure cases.")

 test_index expected_label predicted_label                                                                                                                                                                                  failure_analysis
         12          WORLD        SCI_TECH                The article concerns a bird flu outbreak in Malaysia. The model predicted SCI_TECH instead of WORLD, possibly emphasizing the medical subject over its international-news framing.
         25       BUSINESS        SCI_TECH        The article describes retailers promoting plasma televisions. The model predicted SCI_TECH instead of BUSINESS, possibly emphasizing the technology product over the retail sales context.
         29       SCI_TECH           WORLD The article describes a study about food and water challenges in Southern Africa. The model predicted WORLD instead of SCI_TECH, possibly emphasizing regional affairs over the research framing.

Saved three documented failure cases.


#Stage 6 — Calculate GPU memory for 7B fine-tuning

In [ ]:
import json

GIB = 1024 ** 3
params = 7_000_000_000

# Full fine-tuning: FP16 weights + gradients + AdamW
full_bytes = params * (2 + 2 + 8)
full_with_master_bytes = full_bytes + params * 4

# LoRA: illustrative 32-layer model with
# 4096x4096 query and value projection matrices.
rank = 8
layers = 32
hidden = 4096
target_matrices = 2

adapter_params = (
    layers * target_matrices
    * rank * (hidden + hidden)
)

adapter_training_bytes = adapter_params * (2 + 2 + 8)

# Frozen pretrained weights
fp16_base_bytes = params * 2
four_bit_base_bytes = params * 0.5

results = {
    "full_finetuning_gib": full_bytes / GIB,
    "full_with_fp32_master_gib": full_with_master_bytes / GIB,
    "lora_trainable_parameters": adapter_params,
    "lora_gib": (
        fp16_base_bytes + adapter_training_bytes
    ) / GIB,
    "qlora_gib": (
        four_bit_base_bytes + adapter_training_bytes
    ) / GIB,
}

for key, value in results.items():
    print(f"{key}: {value:,.3f}")

with open(
    "task1_results/memory_estimates.json", "w"
) as f:
    json.dump(results, f, indent=2)

print("\nMemory estimates saved.")

full_finetuning_gib: 78.231
full_with_fp32_master_gib: 104.308
lora_trainable_parameters: 4,194,304.000
lora_gib: 13.085
qlora_gib: 3.307

Memory estimates saved.


#Stage 7A — Back up all Task 1 artifacts

In [ ]:
import os
import json
import shutil
from pathlib import Path
from google.colab import files

ROOT = Path("/content/task1_results")

required_files = [
    "baseline_metrics.json",
    "finetuned_metrics.json",
    "base_predictions.csv",
    "lora_predictions.csv",
    "all_failure_cases.csv",
    "failure_cases.csv",
    "memory_estimates.json",
    "training_logs.json",
    "lora_adapter/adapter_config.json",
]

missing = [
    name for name in required_files
    if not (ROOT / name).exists()
]

assert not missing, f"Missing files: {missing}"

# Load actual recorded results
with open(ROOT / "baseline_metrics.json") as f:
    baseline = json.load(f)

with open(ROOT / "finetuned_metrics.json") as f:
    finetuned = json.load(f)

# Save experiment metadata
metadata = {
    "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
    "dataset": "fancyzhx/ag_news",
    "training_examples": 800,
    "validation_examples": 100,
    "heldout_examples": 100,
    "random_seed": 42,
    "lora_rank": 8,
    "lora_alpha": 16,
    "lora_targets": ["q_proj", "v_proj"],
    "training_epochs": 2,
    "learning_rate": 0.0002,
    "base_accuracy": baseline["accuracy"],
    "lora_accuracy": finetuned["accuracy"],
    "improvement_percentage_points": round(
        (finetuned["accuracy"] - baseline["accuracy"]) * 100,
        2
    ),
    "hardware": "NVIDIA Tesla T4",
    "memory_estimates": "Theoretical parameter-related memory only"
}

with open(ROOT / "experiment_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Create final ZIP with complete directory structure
archive = shutil.make_archive(
    "/content/task1_final_backup",
    "zip",
    root_dir="/content",
    base_dir="task1_results"
)

print("Final backup:", archive)
print("\nIncluded files:")

for path in sorted(ROOT.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(ROOT))

files.download(archive)

Final backup: /content/task1_final_backup.zip

Included files:
- all_failure_cases.csv
- base_predictions.csv
- baseline_metrics.json
- experiment_metadata.json
- failure_cases.csv
- finetuned_metrics.json
- lora_adapter/README.md
- lora_adapter/adapter_config.json
- lora_adapter/adapter_model.safetensors
- lora_adapter/chat_template.jinja
- lora_adapter/tokenizer.json
- lora_adapter/tokenizer_config.json
- lora_predictions.csv
- memory_estimates.json
- training_logs.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>